# Mangrove Pulse: GMW reference sensitivity and WorldCover context


Revision render-fix-2 (2026-10-01) includes the installer repair. To recover a plotting failure in an existing session, run `import os; os.environ["MPLBACKEND"] = "Agg"` in a new cell, then rerun only step 5. Avoid restarting or running all cells again; this preserves existing uploads and dependencies.
Experiment MP-REFERENCE-SENSITIVITY-01 is separate from the original satellite baseline. The [reference-sensitivity guide](../docs/REFERENCE_SENSITIVITY.md) presents the completed results and distinguishes this later repaired source revision from the original run.

Compare GMW v3 and v4.1.12 for 2018 using the exact cached 20 m grid and original review cases C001/C017/C027. WorldCover 2020 supplies later-date context, not new model scores, an ecological diagnosis or drone validation. No threshold is fitted.

## Run in Colab

For a new reproduction, choose Runtime > Run all in a standard CPU runtime.
When the file picker appears, select these two files together:

- `gmw_v4112_2018_mng_ext_cntry_info_vec.gpkg.gz`
- `ESA_WorldCover_10m_2020_v100_N24E054_Map.tif`

Use the two listed datasets, not MTD_MSIL2A.xml, screenshots or a full Sentinel scene. The small frozen reference, audit rows and case register are embedded and checksum-checked.

The final cell downloads a results ZIP, including diagnostics if checks fail. Review its status before interpreting results. If automatic download is blocked, the final cell shows a clickable download link.

The global GMW upload is about 165 MB and expands to about 825 MB in temporary storage. WorldCover is about 21 MB. Only a small local raster window and vector subset are analysed. A dedicated temporary Python environment avoids conflicts with Colab's preloaded libraries. Package installation and upload may take several minutes.

Downloaded originals and frozen PoC outputs remain unchanged.


## What this can establish

The comparison measures map-reference sensitivity at the frozen AOI centres and three previously selected review cases.

Differences mix reference release, source resolution, representation and sampling. They are not confirmed model errors or ecological changes. An unmapped GMW point is not verified absence.

WorldCover 2020 is not a November 2018 validation dataset. Its method uses Sentinel-1/Sentinel-2 and GMW among auxiliary inputs; agreement cannot be treated as fully independent evidence. Its InputQuality layer is not supplied. [WorldCover v100 manual](https://esa-worldcover.s3.eu-central-1.amazonaws.com/v100/2020/docs/WorldCover_PUM_V1.0.pdf)

The newer GMW dataset is also not field truth. [GMW v4.1.12 provider record](https://zenodo.org/records/21346457)

Original threshold 0.49, spatial split/200 m gap, predictions, audit categories and scores remain frozen. No new Sentinel pixels are requested or processed.

Missing, changed, ambiguous or invalid data stop the experiment rather than being interpreted as background. This source revision has no stored execution outputs; the completed earlier experiment is documented separately.


## 1. Create fresh run folders

Only this run's output folder is packaged. Uploaded global datasets and the temporary environment are excluded.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, subprocess, sys, tempfile, traceback, zipfile
from datetime import datetime, timezone

RUN = Path(tempfile.mkdtemp(prefix="mangrove_pulse_reference_"))
ARTIFACTS = RUN / "results"
INPUTS = RUN / "uploads"
SCRATCH = RUN / "scratch"
for path in (ARTIFACTS, INPUTS, SCRATCH, ARTIFACTS / "embedded"):
    path.mkdir(parents=True, exist_ok=True)

STATUS = {"experiment": "MP-REFERENCE-SENSITIVITY-01", "status": "initializing",
          "started_utc": datetime.now(timezone.utc).isoformat(),
          "execution_location": "Team-run Google Colab", "notebook_revision": "render-fix-2", "scientific_results_verified_locally": False}
DEPS_OK = UPLOAD_OK = False
def save_status():
    (ARTIFACTS / "run_status.json").write_text(json.dumps(STATUS, indent=2) + "\n", encoding="utf-8")
save_status()
print("Fresh run:", RUN.name)


## 2. Restore checksum-verified frozen inputs and reproducible source

This cell does not fit or execute the original baseline.

In [ ]:
EMBEDDED = json.loads("{\"audit_cells_original.csv\":{\"workspace_source\":\"submission/examples/audit/all_audit_cells.csv\",\"base64\":\"...\",\"sha256\":\"80a92cf778a92f8bdcc4a99e39dfe906f193a3eaad1dacbbd166f13481806e73\"},\"gmw3.tif\":{\"workspace_source\":\"reproduction/reproduction_c80aaac7/baseline/inputs/previous/gmw2018_on_scl_grid.tif\",\"base64\":\"SUkqAJQEAAASAAABAwABAAAAegAAAAEBAwABAAAAcAAAAAIBAwABAAAACAAAAAMBAwABAAAACAAAAAYBAwABAAAAAQAAABUBAwABAAAAAQAAABwBAwABAAAAAgAAAD0BAwABAAAAAQAAAEIBAwABAAAAAAIAAEMBAwABAAAAAAIAAEQBBAABAAAAAAAAAEUBBAABAAAAAAAAAFMBAwABAAAAAQAAAA6DDAADAAAA5gAAAIKEDAAGAAAA/gAAAK+HAwAgAAAALgEAALGHAgAeAAAAbgEAAIGkAgAEAAAAMjU1AAAAAAAAAAAAAAA0QAAAAAAAADRAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgIMaQQAAAADqGUVBAAAAAAAAAAABAAEAAAAHAAAEAAABAAEAAQQAAAEAAQACBLGHFgAAAAEIsYcHABYABggAAAEAjiMADAAAAQCAfwQMAAABACkjV0dTIDg0IC8gVVRNIHpvbmUgNDBOfFdHUyA4NHwAeJzt3MFy4jAQRVH1/380syRMbEmkmJqo3zmrVEiRxW0bYRkej/9l8Au8naGqZo/V4CQf7T/0P81n+w/9D/MYb1bTv5XHGO9lW5ziDcBZPt5/cJL334bp34n+2X7Sf7oAHJzk4/0NQG9lARhO/2xO8dlc5M+mfza7vNn0D6d/tnIGiKZ/Nv2z6Z9tfgvwoLnZ4a9/f/pnm/S3MAiw198kdHXfv/RPoH+0+fH/HIBBSx/p7/Pfx7rrX6V/gs3+c/qf6yZ0Pa2fQ/9z3TTWP4T+2S4bV+kf4aZxvTcA+p9K/2zr1b/+nemfbdl/a/NH/0PdHt/6J6hl/zF2Nv/0P9Tk5f3LQ8sTgP6Hmvd//rh4Gv0PpX+22rqzp6rmdwPr31pVjekyUP/Wlm8D9W9teQ1Q/9b0z6Z/tuUmkP6NbWwC6t/Yxi6w/n2V/sm2bgLRv61y/Efbug9M/672bgTUv6nSP5r+4fYGQP+uXuvr39P0O2CranURWP+jzV7aq6rG7JMCQ//O9M/22v9mAvRva+sNgP6nm34J+HIA9O9r5wSgf1/6J6vSP5r+4az/Y9zkXU+A/p193QDQP89L/8u/0L+J2VfB659K/2zWf0Hq4lf659CfF67/Zaty/Oeq0j9Z7QyA/m3pH630T1b6R9M/W+0NgP5NvUTXP47+4f5Orn+Wq/7fB0D/vl576x9O/2z6Z7t8AdA/hv7pLiZA/yAXF4H0D6J/Nv3DfR8A/ZPon835P5v+v9TjbT/7P/oDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADAv/cHDsAKjwATAAABAwABAAAAegAAAAEBAwABAAAAcAAAAAIBAwABAAAACAAAAAMBAwABAAAACAAAAAYBAwABAAAAAQAAABUBAwABAAAAAQAAABwBAwABAAAAAgAAAD0BAwABAAAAAQAAAEIBAwABAAAAAAIAAEMBAwABAAAAAAIAAEQBBAABAAAAjAEAAEUBBAABAAAABwMAAFMBAwABAAAAAQAAAA6DDAADAAAAfgUAAIKEDAAGAAAAlgUAAK+HAwAgAAAAxgUAALGHAgAeAAAABgYAAICkAgC3AAAAJAYAAIGkAgAEAAAAMjU1AAAAAAAAAAAAAAA0QAAAAAAAADRAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgIMaQQAAAADqGUVBAAAAAAAAAAABAAEAAAAHAAAEAAABAAEAAQQAAAEAAQACBLGHFgAAAAEIsYcHABYABggAAAEAjiMADAAAAQCAfwQMAAABACkjV0dTIDg0IC8gVVRNIHpvbmUgNDBOfFdHUyA4NHwAPEdEQUxNZXRhZGF0YT4KICA8SXRlbSBuYW1lPSJkZXNjcmlwdGlvbiI+MD1ub3QgbWFwcGVkIGFzIG1hbmdyb3ZlOyAxPUdNVyBtYXBwZWQgbWFuZ3JvdmU7IDI1NT1vdXRzaWRlIEFPSTwvSXRlbT4KICA8SXRlbSBuYW1lPSJzYW1wbGluZyI+bmVhcmVzdCBwaXhlbCBjZW50cmU8L0l0ZW0+CjwvR0RBTE1ldGFkYXRhPgoA\",\"sha256\":\"5613133dcd81fc4961bc8a4152afd127b8a0a250869b37dd5465c0aeb1fd63e0\"},\"AOI.geojson\":{\"workspace_source\":\"outputs/data_access/KHOR_KALBA_AOI.geojson\",\"base64\":\"ewogICJ0eXBlIjogIkZlYXR1cmVDb2xsZWN0aW9uIiwKICAiYmJveCI6IFsKICAgIDU2LjM1LAogICAgMjQuOTg2LAogICAgNTYuMzc0LAogICAgMjUuMDA2CiAgXSwKICAiZmVhdHVyZXMiOiBbCiAgICB7CiAgICAgICJ0eXBlIjogIkZlYXR1cmUiLAogICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAibmFtZSI6ICJNYW5ncm92ZSBQdWxzZSBLaG9yIEthbGJhIHByb3Zpc2lvbmFsIHN0dWR5IHJlY3RhbmdsZSIsCiAgICAgICAgImNvb3JkaW5hdGVfb3JkZXIiOiAibG9uZ2l0dWRlLCBsYXRpdHVkZSIsCiAgICAgICAgImJvdW5kYXJ5X3N0YXR1cyI6ICJTdHVkeSB3aW5kb3cgb25seTsgbm90IGFuIG9mZmljaWFsIHJlc2VydmUsIGhhYml0YXQgb3Igc3VydmV5IGJvdW5kYXJ5IiwKICAgICAgICAic291cmNlIjogIkZyb3plbiBNYW5ncm92ZSBQdWxzZSBleHBlcmltZW50IHJlY29yZGVkIGluIFBST0pFQ1RfTUVNT1JZLm1kIgogICAgICB9LAogICAgICAiZ2VvbWV0cnkiOiB7CiAgICAgICAgInR5cGUiOiAiUG9seWdvbiIsCiAgICAgICAgImNvb3JkaW5hdGVzIjogWwogICAgICAgICAgWwogICAgICAgICAgICBbCiAgICAgICAgICAgICAgNTYuMzUsCiAgICAgICAgICAgICAgMjQuOTg2CiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA1Ni4zNzQsCiAgICAgICAgICAgICAgMjQuOTg2CiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA1Ni4zNzQsCiAgICAgICAgICAgICAgMjUuMDA2CiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA1Ni4zNSwKICAgICAgICAgICAgICAyNS4wMDYKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDU2LjM1LAogICAgICAgICAgICAgIDI0Ljk4NgogICAgICAgICAgICBdCiAgICAgICAgICBdCiAgICAgICAgXQogICAgICB9CiAgICB9CiAgXQp9Cg==\",\"sha256\":\"2ee5972b8398ad0e35ece10a0ace5037e6bf032cfee33a6446661c1170852596\"},\"frozen_experiment_plan.json\":{\"workspace_source\":\"submission/examples/baseline/frozen_experiment_plan.json\",\"base64\":\"ewogICJtZXRob2QiOiAiT25lIHNwYXRpYWwgaG9sZG91dDsgbm9ydGhlcm4gY2FsaWJyYXRpb24gYW5kIHNvdXRoZXJuIGV2YWx1YXRpb24iLAogICJjdXRfbm9ydGhpbmdfbSI6IDI3NjQ2NTAuMCwKICAiZXhjbHVkZWRfaGFsZl9nYXBfbSI6IDEwMCwKICAiZ2FwX3RvdGFsX20iOiAyMDAsCiAgImNhbGlicmF0aW9uX2NlbnRyZXNfYmVmb3JlX3F1YWxpdHkiOiA2MTkxLAogICJldmFsdWF0aW9uX2NlbnRyZXNfYmVmb3JlX3F1YWxpdHkiOiA2MTIxLAogICJmZWF0dXJlIjogIihtZWFuIHJhdyBCMDggLSBtZWFuIHJhdyBCMDQpLyhtZWFuIHJhdyBCMDggKyBtZWFuIHJhdyBCMDQpLCAyeDIgbmF0aXZlIHBpeGVscyIsCiAgImZlYXR1cmVfY2F2ZWF0IjogIk5vcm1hbGl6ZWQgcmVkL05JUiBkaWdpdGFsLXZhbHVlIGNvbnRyYXN0LCBub3QgYXNzZXJ0ZWQgY2FsaWJyYXRlZCBORFZJIG9yIGEgaGVhbHRoIG1lYXN1cmVtZW50LiIsCiAgInRocmVzaG9sZF9jYW5kaWRhdGVzIjogIi0xLjAwIHRocm91Z2ggMS4wMCwgc3RlcCAwLjAxOyBwcmVkaWN0IG1hbmdyb3ZlIGlmIGNvbnRyYXN0ID49IHRocmVzaG9sZCIsCiAgInNlbGVjdGlvbiI6ICJNYXhpbXVtIGNhbGlicmF0aW9uIEYxIGFnYWluc3QgR01XOyB0aWVzIGNob29zZSBoaWdoZXN0IHRocmVzaG9sZCIsCiAgInF1YWxpdHkiOiAiU0NMIDQsNSw2IGFuZCBmb3VyIHZhbGlkIG5vbnplcm8gc3VicGl4ZWxzIGluIGVhY2ggYmFuZDsgd2F0ZXIgcmV0YWluZWQiLAogICJtaW5pbXVtX2VhY2hfY2xhc3NfcGVyX3NwbGl0IjogNTAsCiAgIm1pbmltdW1fbm90ZSI6ICJQcmFjdGljYWwgc3RvcCBydWxlLCBub3QgYSBzdGF0aXN0aWNhbCBhZGVxdWFjeSBndWFyYW50ZWUuIiwKICAicHJpbWFyeV9kYXRlIjogIjIwMTgtMTEtMjAsIHJldGFpbmVkIGJlZm9yZSBmaXR0aW5nOyBOb3ZlbWJlciAzMCBpcyBhIGZpeGVkLXRocmVzaG9sZCBzZW5zaXRpdml0eSBjaGVjayIsCiAgImV2YWx1YXRpb25fcnVsZXMiOiAiTm8gdGhyZXNob2xkIHJldHVuaW5nIG9yIHNwbGl0IGNoYW5nZXMgYmFzZWQgb24gc291dGhlcm4gcmVzdWx0cy4iCn0=\",\"sha256\":\"4b7e92ff7af7167d8cadc59e97cfd1b7d3f09cf714afa975981f32bf509a293f\"},\"case_reviews_original.json\":{\"workspace_source\":\"submission/examples/review/case_reviews.json\",\"base64\":\"WwogIHsKICAgICJjYXNlIjogIkMwMDEiLAogICAgImNlbGxzIjogMTEsCiAgICAiY2F0ZWdvcnkiOiAiUHJlZGljdGlvbiBwb3NpdGl2ZSAvIEdNVyBub3QgbWFwcGVkIiwKICAgICJvYnNlcnZhdGlvbiI6ICJDZWxscyBmb2xsb3cgYSB2ZWdldGF0aW9uIGVkZ2UgYW5kIG92ZXJsYXAgY2Fub3B5LWxpa2UgdGV4dHVyZS4iLAogICAgImludGVycHJldGF0aW9uIjogIlBvc3NpYmxlIHZlZ2V0YXRpb24gYmV5b25kIG1hcHBlZCBleHRlbnQ7IG1hbmdyb3ZlIGlkZW50aXR5IGFuZCBpbmRpdmlkdWFsLWNlbGwgdHJ1dGggdW52ZXJpZmllZC4iLAogICAgIm5leHRfZXZpZGVuY2UiOiAiRGF0ZWQgY292ZXIgbGFiZWxzIGFuZCBtZWFzdXJlZCBsYXllciBhbGlnbm1lbnQuIiwKICAgICJzY3JlZW5zaG90IjogIlNjcmVlbnNob3QgKDI3KS5wbmciLAogICAgInJldmlld19zdGF0dXMiOiAiUXVhbGl0YXRpdmUgcmV2aWV3IGNvbXBsZXRlZDsgbGFiZWwgdHJ1dGggdW5yZXNvbHZlZCIsCiAgICAiY2FwdHVyZV9kYXRlIjogIjIwMTgtMTAtMDUiLAogICAgImFyY2hpdmVfZGF0ZSI6ICIyMDIwLTEwLTE0IiwKICAgICJuYXRpdmVfem9vbSI6IDE3LAogICAgImZpZWxkX2FjY2VzcyI6ICJ1bmNvbmZpcm1lZCIKICB9LAogIHsKICAgICJjYXNlIjogIkMwMTciLAogICAgImNlbGxzIjogMzIsCiAgICAiY2F0ZWdvcnkiOiAiUHJlZGljdGlvbiBuZWdhdGl2ZSAvIEdNVyBtYXBwZWQiLAogICAgIm9ic2VydmF0aW9uIjogIkNlbGxzIGNyb3NzIHBhdGNoeSB2ZWdldGF0aW9uIGFuZCBsaWdodGVyIG9wZW4gY292ZXIuIiwKICAgICJpbnRlcnByZXRhdGlvbiI6ICJTcGFyc2Ugb3IgbWl4ZWQgY292ZXIgaXMgcGxhdXNpYmxlOyBub3QgMzIgY29uZmlybWVkIG1pc3NlZCBtYW5ncm92ZSBjZWxscy4iLAogICAgIm5leHRfZXZpZGVuY2UiOiAiSW5kZXBlbmRlbnQgZGF0ZWQgY2Fub3B5IGxhYmVsczsgYWNjb3VudCBmb3IgbWl4ZWQgY2VsbHMuIiwKICAgICJzY3JlZW5zaG90IjogIlNjcmVlbnNob3QgKDI4KS5wbmciLAogICAgInJldmlld19zdGF0dXMiOiAiUXVhbGl0YXRpdmUgcmV2aWV3IGNvbXBsZXRlZDsgbGFiZWwgdHJ1dGggdW5yZXNvbHZlZCIsCiAgICAiY2FwdHVyZV9kYXRlIjogIjIwMTgtMTAtMDUiLAogICAgImFyY2hpdmVfZGF0ZSI6ICIyMDIwLTEwLTE0IiwKICAgICJuYXRpdmVfem9vbSI6IDE3LAogICAgImZpZWxkX2FjY2VzcyI6ICJ1bmNvbmZpcm1lZCIKICB9LAogIHsKICAgICJjYXNlIjogIkMwMjciLAogICAgImNlbGxzIjogNDIsCiAgICAiY2F0ZWdvcnkiOiAiUXVhbGl0eS1leGNsdWRlZCAvIEdNVyBtYXBwZWQiLAogICAgIm9ic2VydmF0aW9uIjogIkxvbmcgc3RyaXAgZm9sbG93cyBhIGRhcmsgY2hhbm5lbCBhbmQgdmVnZXRhdGlvbiBtYXJnaW4uIiwKICAgICJpbnRlcnByZXRhdGlvbiI6ICJRdWFsaXR5IGFuZCBjb3ZlcmFnZSBnYXA7IHNjcmVlbmluZyBjYXVzZSBhbmQgY2VsbCBjb3ZlciByZW1haW4gdW5yZXNvbHZlZC4iLAogICAgIm5leHRfZXZpZGVuY2UiOiAiUmV2aWV3IFNDTCwgaW5kZXBlbmRlbnQgY292ZXIgbGFiZWxzIGFuZCBhbGlnbm1lbnQ7IGRvIG5vdCBhc3NpZ24gYSB0aWRhbCBjYXVzZS4iLAogICAgInNjcmVlbnNob3QiOiAiU2NyZWVuc2hvdCAoMjkpLnBuZyIsCiAgICAicmV2aWV3X3N0YXR1cyI6ICJRdWFsaXRhdGl2ZSByZXZpZXcgY29tcGxldGVkOyBsYWJlbCB0cnV0aCB1bnJlc29sdmVkIiwKICAgICJjYXB0dXJlX2RhdGUiOiAiMjAxOC0xMC0wNSIsCiAgICAiYXJjaGl2ZV9kYXRlIjogIjIwMjAtMTAtMTQiLAogICAgIm5hdGl2ZV96b29tIjogMTcsCiAgICAiZmllbGRfYWNjZXNzIjogInVuY29uZmlybWVkIgogIH0KXQo=\",\"sha256\":\"7c106f1ce89934c622b371f66537b43f0f0db0e22dd73aebb528b05e78d3f109\"}}")
MODULE_SOURCE = "\"\"\"MP-REFERENCE-SENSITIVITY-01: separate team-run Colab experiment.\nNo model fitting, EO acquisition, ecological diagnosis or change attribution.\n\"\"\"\nimport csv\nimport gzip\nimport hashlib\nimport json\nimport platform\nimport shutil\nfrom pathlib import Path\n\nEXPERIMENT = \"MP-REFERENCE-SENSITIVITY-01\"\nAOI = [56.350, 24.986, 56.374, 25.006]\nEXPECTED = {\n    \"gmw_v4112_2018_mng_ext_cntry_info_vec.gpkg.gz\": {\n        \"size\": 165428975,\n        \"sha256\": \"53eda1adad353cc349ae840cfba7e934dc7e2651769491bc2b367dcf2b597f96\",\n    },\n    \"ESA_WorldCover_10m_2020_v100_N24E054_Map.tif\": {\n        \"size\": 21292309,\n        \"sha256\": \"8ae908a79184d0bdf7a9a2335b5c7a0366e9e52a1c7197c4c926c38b880d1536\",\n    },\n}\nWC_LABELS = {\n    0: \"No data\", 10: \"Tree cover\", 20: \"Shrubland\", 30: \"Grassland\",\n    40: \"Cropland\", 50: \"Built-up\", 60: \"Bare / sparse vegetation\",\n    70: \"Snow / ice\", 80: \"Permanent water\", 90: \"Herbaceous wetland\",\n    95: \"Mangroves\", 100: \"Moss / lichen\",\n}\nLIMITATIONS = [\n    \"Reference-map sensitivity, not ecological or field accuracy.\",\n    \"GMW epochs represent annual extents, not exact November observation dates.\",\n    \"GMW v3 cached nearest-pixel labels and v4.1.12 vector point membership differ in release, source resolution and representation; differences cannot be attributed to one cause.\",\n    \"Both GMW releases belong to the same product family; v4.1.12 uses optical and SAR imagery, including Sentinel-2. They are not independent field truth.\",\n    \"WorldCover is later-2020 context, not validation of November 2018. Differences do not demonstrate growth, loss, tide, stress or map error.\",\n    \"WorldCover uses Sentinel-1/Sentinel-2 and GMW among auxiliary inputs; independence cannot be assumed.\",\n    \"WorldCover InputQuality is not uploaded. Class values do not supply local certainty.\",\n    \"Zero GMW means not mapped, not proven absence. WorldCover zero is missing data, not background.\",\n    \"Counts describe sampled 20 m grid centres, not native pixel counts, canopy fractions or confirmed mangrove area.\",\n    \"Point membership includes polygon boundaries. No all-touched burn, alignment shifts, threshold retuning or split change.\",\n    \"Review cases were already selected and inspected; comparisons are descriptive/post-hoc, not fresh held-out validation.\",\n    \"No field observations, drone operations, planting needs, survival or carbon benefit are established.\",\n]\nATTRIBUTION = \"\"\"# Sources and attribution\n\nGMW v3: Bunting, P.; Rosenqvist, A.; Hilarides, L.; Lucas, R.; Thomas, N.;\nTadono, T.; Worthington, T.; Spalding, M.; Murray, N.; Rebelo, L.-M. (2022).\nGlobal Mangrove Watch (1996-2020), Version 3.0.\nhttps://doi.org/10.5281/zenodo.6894273\nCC BY 4.0. Cached nearest-centre 2018 labels are embedded unchanged.\nThe original reference preparation sampled cropped source tiles onto the SCL grid.\n\nGMW v4.1.12: Bunting, P.; Hilarides, L.; Rosenqvist, A.; Oakes, G.;\nCastro Brandao Vaz dos Santos, P.; Worthington, T.; Rosenqvist, J.;\nZanaga, D.; Van De Kerchove, R.; Tadono, T.; Kobayashi, K. (2026).\nGlobal Mangrove Watch: Timeseries of Mangrove Extent, version 4.1.12.\nhttps://doi.org/10.5281/zenodo.21346457\nCC BY 4.0, confirmed through the public Zenodo record API on 2026-09-30.\nModifications: AOI spatial filter, projection, clipping and grid-centre sampling.\n\nWorldCover: Zanaga, D. et al. (2021). ESA WorldCover 10 m 2020 v100.\nhttps://doi.org/10.5281/zenodo.5571936\nCC BY 4.0. Attribution: © ESA WorldCover project 2020 /\nContains modified Copernicus Sentinel data (2020) processed by ESA WorldCover consortium.\nModifications: small native-grid crop and categorical centre sampling.\n\nNo endorsement is implied. Team case coordinates, original audit fields and\nreview notes are cached PoC evidence; notes are not field labels.\nNo third-party imagery pixels or team's basemap screenshots are embedded.\n\nPrimary documentation:\n- https://esa-worldcover.org/en/data-access\n- https://esa-worldcover.s3.eu-central-1.amazonaws.com/v100/2020/docs/WorldCover_PUM_V1.0.pdf\n- https://pyogrio.readthedocs.io/en/latest/api.html\n- https://shapely.readthedocs.io/en/stable/reference/shapely.intersects_xy.html\n- https://rasterio.readthedocs.io/en/stable/topics/windowed-rw.html\n\"\"\"\nMETHOD = \"\"\"# Experiment plan: MP-REFERENCE-SENSITIVITY-01\n\nStatus: unexecuted until the team runs Colab. Created 2026-10-01.\nPurpose: describe reference-map sensitivity for the frozen Khor Kalba PoC and\nthree already-reviewed cases; add clearly later-date 2020 land-cover context.\n\nPreserve the cached GMW v3 raster/grid, original audit rows and case register.\nUse all 13,401 AOI centres for the GMW map cross-tabulation, without a Sentinel\nquality gate. This population differs from the original eligible southern\nevaluation; no new model score is computed. Original threshold 0.49 and\nnorth/south split with a 200 m gap remain unchanged.\n\nCompare GMW v3 2018 with GMW v4.1.12 2018 on the EXACT cached EPSG:32640,\n20 m centre coordinates. v4 labels are point intersections with polygons:\nboundaries included, any polygon counts as mapped. Select a unique polygon\nlayer, transform the spatial-query bounds into its CRS, read only local\nfeatures, reject missing/invalid geometries rather than repairing silently,\nproject to the grid CRS, then clip to the grid bounds.\n\nWorldCover v100 represents 2020. Read only a small native EPSG:4326 window\ncovering the frozen grid. For every grid centre, transform x/y to longitude/\nlatitude with always_xy=True and use the containing native raster pixel\n(equivalent to nearest-centre sampling on this regular raster, subject to\nties at pixel boundaries). Never interpolate categorical class codes.\nMasked native values become zero (missing), explicitly counted separately.\nNo source quality layer is supplied.\n\nReport: GMW 2x2 reference cross-tab and Jaccard, full-AOI sampled WorldCover\nclass histogram, paired summaries for C001/C017/C027, and all 208 original\naudit centres with added map values. Jaccard is map agreement, not accuracy.\nNo feature fitting, threshold search, shifting, field labels or causation.\n\nThese are release/representation/sampling comparisons. v3 and v4 native\nsources have different representations/resolutions; differences alone cannot\nisolate a release algorithm effect. WorldCover also uses GMW auxiliary input.\n\nChecks: strict input sizes and SHA-256, embedded-file hashes, frozen raster\nfootprint, native WorldCover metadata/classes/coverage, unique case IDs and\nexpected case populations, audit label/coordinate consistency, synthetic\nhelper tests, same-grid output invariants and archive CRC/hash manifest.\nInvalid/missing/ambiguous inputs stop the experiment and still produce a\ndiagnostic ZIP when the Colab runtime reaches the packaging cell.\n\nZIP includes source, tests, small embedded inputs, local clipped maps,\nper-centre JSON, summaries, PNGs, methods, citations, environment and logs.\nIt excludes uploaded global datasets and private account information.\nDo not publish/add to submission before inspecting returned results.\n\"\"\"\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\"rb\") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\"\"):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef dump(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \"\\n\", encoding=\"utf-8\")\n\n\ndef verify_input(path, expected):\n    if Path(path).stat().st_size != expected[\"size\"] or sha256(path) != expected[\"sha256\"]:\n        raise ValueError(\"Unexpected or changed input: \" + Path(path).name)\n\n\ndef compare_binary(old, new, valid):\n    import numpy as np\n    old, new, valid = np.asarray(old), np.asarray(new), np.asarray(valid, bool)\n    if old.shape != new.shape or old.shape != valid.shape:\n        raise ValueError(\"Comparison grids differ.\")\n    if not np.isin(old[valid], [0, 1]).all() or not np.isin(new[valid], [0, 1]).all():\n        raise ValueError(\"Nonbinary values in compared cells.\")\n    counts = {\n        \"both_not_mapped\": int(np.sum(valid & (old == 0) & (new == 0))),\n        \"v3_only_mapped\": int(np.sum(valid & (old == 1) & (new == 0))),\n        \"v4_only_mapped\": int(np.sum(valid & (old == 0) & (new == 1))),\n        \"both_mapped\": int(np.sum(valid & (old == 1) & (new == 1))),\n    }\n    union = sum(counts[k] for k in (\"v3_only_mapped\", \"v4_only_mapped\", \"both_mapped\"))\n    total = int(valid.sum())\n    changed = counts[\"v3_only_mapped\"] + counts[\"v4_only_mapped\"]\n    return dict(counts, centres=total, differently_mapped_centres=changed,\n                differently_mapped_fraction=changed / total if total else None,\n                mapped_jaccard=counts[\"both_mapped\"] / union if union else None,\n                interpretation=\"Reference-map agreement, not field accuracy or ecological change\")\n\n\ndef class_counts(values):\n    import numpy as np\n    vals, counts = np.unique(values, return_counts=True)\n    if not set(int(v) for v in vals).issubset(WC_LABELS):\n        raise ValueError(\"Unexpected WorldCover code.\")\n    return [{\"code\": int(v), \"class\": WC_LABELS[int(v)], \"centres\": int(n)}\n            for v, n in zip(vals, counts)]\n\n\ndef polygon_labels(geometries, xs, ys):\n    import numpy as np\n    import shapely\n    if not geometries:\n        raise ValueError(\"No local GMW polygons: do not silently infer absence.\")\n    for geom in geometries:\n        if geom is None or geom.is_empty or not geom.is_valid:\n            raise ValueError(\"Empty/missing/invalid local geometry. No automatic repair.\")\n        if geom.geom_type not in (\"Polygon\", \"MultiPolygon\"):\n            raise ValueError(\"Nonpolygon geometry.\")\n    return np.asarray(shapely.intersects_xy(shapely.union_all(geometries), xs, ys), dtype=\"uint8\")\n\n\ndef sample_native(array, transform, xs, ys):\n    import numpy as np\n    xs, ys = np.asarray(xs), np.asarray(ys)\n    if xs.shape != ys.shape:\n        raise ValueError(\"Coordinate array shapes differ.\")\n    c, r = (~transform) * (xs, ys)\n    c, r = np.floor(c).astype(int), np.floor(r).astype(int)\n    within = (r >= 0) & (c >= 0) & (r < array.shape[0]) & (c < array.shape[1])\n    sampled = np.zeros(xs.shape, dtype=\"uint8\")\n    sampled[within] = array[r[within], c[within]]\n    return sampled, within\n\n\ndef write_grid(path, array, profile, description, **tags):\n    import rasterio\n    prof = profile.copy()\n    prof.update(count=1, dtype=\"uint8\", nodata=255, compress=\"deflate\")\n    with rasterio.open(path, \"w\", **prof) as ds:\n        ds.write(array.astype(\"uint8\"), 1)\n        ds.update_tags(experiment=EXPERIMENT, description=description, **tags)\n\n\ndef run(inputs, embedded, out, scratch, layer_name=None):\n    import numpy as np\n    import rasterio\n    import geopandas as gpd\n    import pyogrio\n    import pyproj\n    import shapely\n    from rasterio.features import geometry_window\n    from rasterio.warp import transform_bounds\n    from shapely.geometry import box, mapping\n    out, inputs, embedded, scratch = map(Path, (out, inputs, embedded, scratch))\n    checks = []\n\n    def check(name, condition):\n        checks.append({\"check\": name, \"passed\": bool(condition)})\n        dump(out / \"data_checks.json\", checks)\n        if not condition:\n            raise ValueError(\"Data check failed: \" + name)\n\n    dump(out / \"environment.json\", {\n        \"python\": platform.python_version(), \"numpy\": np.__version__,\n        \"rasterio\": rasterio.__version__, \"rasterio_gdal\": rasterio.__gdal_version__,\n        \"geopandas\": gpd.__version__, \"pyogrio\": pyogrio.__version__,\n        \"pyogrio_gdal\": pyogrio.__gdal_version_string__, \"pyproj\": pyproj.__version__,\n        \"shapely\": shapely.__version__,\n    })\n    for name, expected in EXPECTED.items():\n        verify_input(inputs / name, expected)\n        check(\"Uploaded checksum: \" + name, True)\n    with rasterio.open(embedded / \"gmw3.tif\") as ds:\n        old, tx, crs, profile = ds.read(1), ds.transform, ds.crs, ds.profile.copy()\n        bounds = ds.bounds\n        check(\"Frozen north-up 20 m EPSG:32640 grid\",\n              crs.to_epsg() == 32640 and tx.a == 20 and tx.e == -20 and tx.b == 0 and tx.d == 0)\n        grid = {\"shape\": list(old.shape), \"transform\": list(tx)[:6], \"crs\": str(crs),\n                \"bounds\": list(bounds), \"reference_sha256\": sha256(embedded / \"gmw3.tif\")}\n    inside = old != 255\n    check(\"Frozen reference codes and counts\",\n          set(np.unique(old)).issubset({0, 1, 255}) and inside.sum() == 13401 and (old == 1).sum() == 1158)\n    rr, cc = np.indices(old.shape)\n    xs, ys = tx * (cc + .5, rr + .5)\n    lon, lat = pyproj.Transformer.from_crs(crs, \"EPSG:4326\", always_xy=True).transform(xs, ys)\n    dump(out / \"grid.json\", grid)\n    source_aoi = json.loads((embedded / \"AOI.geojson\").read_text(encoding=\"utf-8\"))\n    check(\"AOI bbox unchanged\", source_aoi.get(\"bbox\") == AOI)\n\n    gpkg = scratch / \"gmw2018.gpkg\"\n    with gzip.open(inputs / \"gmw_v4112_2018_mng_ext_cntry_info_vec.gpkg.gz\", \"rb\") as src, gpkg.open(\"wb\") as dst:\n        shutil.copyfileobj(src, dst, length=1024 * 1024)\n    check(\"Full gzip CRC/length\", gpkg.stat().st_size == 824799232)\n    layers = pyogrio.list_layers(gpkg).tolist()\n    dump(out / \"gmw_layers.json\", {\"layers\": layers})\n    polygons = [name for name, typ in layers if typ and \"Polygon\" in typ]\n    if layer_name is not None:\n        check(\"Explicit polygon layer exists\", layer_name in polygons)\n    else:\n        check(\"One unambiguous polygon layer\", len(polygons) == 1)\n        layer_name = polygons[0]\n    info = pyogrio.read_info(gpkg, layer=layer_name)\n    dump(out / \"gmw_layer_info.json\", {\n        \"layer\": layer_name, \"crs\": info[\"crs\"], \"geometry_type\": info[\"geometry_type\"],\n        \"features\": int(info[\"features\"]), \"fields\": info[\"fields\"].tolist(),\n        \"total_bounds\": list(info[\"total_bounds\"]) if info[\"total_bounds\"] is not None else None,\n    })\n    check(\"GMW CRS exists\", bool(info[\"crs\"]))\n    # Query a small buffer beyond the grid; bbox must use the source layer CRS.\n    query_box = transform_bounds(crs, info[\"crs\"],\n                                 bounds.left - 5, bounds.bottom - 5, bounds.right + 5, bounds.top + 5,\n                                 densify_pts=21)\n    local = pyogrio.read_dataframe(gpkg, layer=layer_name, columns=[], bbox=query_box,\n                                  force_2d=True, on_invalid=\"raise\")\n    check(\"Local GMW polygons present\", len(local) > 0)\n    check(\"All source local geometries present and valid\",\n          not local.geometry.isna().any() and not local.geometry.is_empty.any() and local.geometry.is_valid.all())\n    local = local.to_crs(crs)\n    check(\"Projected local geometries valid\", local.geometry.is_valid.all())\n    clipped = [geom.intersection(box(*bounds)) for geom in local.geometry]\n    clipped = [geom for geom in clipped if not geom.is_empty]\n    new = polygon_labels(clipped, xs, ys)\n    new[~inside] = 255\n    check(\"New GMW grid retains exactly frozen AOI\", np.array_equal(new != 255, inside))\n    dump(out / \"gmw_local_diagnostics.json\", {\n        \"queried_features\": len(local), \"clipped_nonempty\": len(clipped),\n        \"invalid_source_geometries\": 0, \"automatic_repairs\": 0,\n        \"query_bbox_layer_crs\": list(query_box), \"point_boundary_rule\": \"intersects, boundary included\",\n        \"note\": \"Local polygon presence is confirmed; unmapped areas are not proven absence.\",\n    })\n    clipped_wgs = gpd.GeoSeries(clipped, crs=crs).to_crs(4326)\n    dump(out / \"gmw4_2018_clipped.geojson\", {\n        \"type\": \"FeatureCollection\",\n        \"features\": [{\"type\": \"Feature\", \"properties\": {\"source\": \"GMW v4.1.12\", \"epoch\": 2018},\n                      \"geometry\": mapping(geom)} for geom in clipped_wgs],\n    })\n    write_grid(out / \"gmw4_2018_on_frozen_grid.tif\", new, profile,\n               \"1=mapped, 0=not mapped, 255=outside frozen AOI; point boundary included\", epoch=\"2018\")\n\n    wc_path = inputs / \"ESA_WorldCover_10m_2020_v100_N24E054_Map.tif\"\n    with rasterio.open(wc_path) as ds:\n        check(\"WorldCover exact native grid metadata\",\n              ds.crs.to_epsg() == 4326 and ds.shape == (36000, 36000) and ds.count == 1\n              and np.allclose(ds.res, [1 / 12000, 1 / 12000], rtol=0, atol=1e-12)\n              and np.allclose(list(ds.bounds), [54, 24, 57, 27]))\n        crop_bounds = transform_bounds(crs, ds.crs, *bounds, densify_pts=21)\n        win = geometry_window(ds, [mapping(box(*crop_bounds))], pad_x=2, pad_y=2)\n        check(\"Small native read only\", win.width * win.height < 2000000)\n        native_masked = ds.read(1, window=win, masked=True)\n        native = native_masked.filled(0).astype(\"uint8\")\n        native_tx = ds.window_transform(win)\n        cp = ds.profile.copy()\n        cp.update(width=native.shape[1], height=native.shape[0], transform=native_tx,\n                  count=1, compress=\"deflate\")\n        native_cmap = ds.colormap(1)\n        wc_meta = {\"source\": wc_path.name, \"epoch\": 2020, \"version\": \"v100\",\n                   \"native_crs\": str(ds.crs), \"native_resolution_degrees\": list(ds.res),\n                   \"window\": [win.col_off, win.row_off, win.width, win.height],\n                   \"native_masked_pixels_in_crop\": int(np.ma.getmaskarray(native_masked).sum()),\n                   \"source_nodata\": ds.nodata, \"input_quality_available\": False}\n    check(\"WorldCover class codes valid in crop\", set(np.unique(native)).issubset(WC_LABELS))\n    with rasterio.open(out / \"worldcover2020_native_crop.tif\", \"w\", **cp) as ds:\n        ds.write(native, 1)\n        ds.write_colormap(1, native_cmap)\n        ds.update_tags(epoch=\"2020\", purpose=\"Later-date context, not 2018 field truth\")\n    wc, within = sample_native(native, native_tx, lon, lat)\n    check(\"WorldCover spatial coverage of AOI\", bool(within[inside].all()))\n    wc_meta[\"aoi_missing_centres\"] = int(((wc == 0) & inside).sum())\n    wc[~inside] = 255\n    check(\"WorldCover sampled grid retains frozen AOI\", np.array_equal(wc != 255, inside))\n    dump(out / \"worldcover_metadata.json\", wc_meta)\n    write_grid(out / \"worldcover2020_on_frozen_grid.tif\", wc, profile,\n               \"2020 context; native-pixel categorical centre sample; 0=missing,255=outside AOI\", epoch=\"2020\")\n\n    original_rows = list(csv.DictReader((embedded / \"audit_cells_original.csv\").read_text(encoding=\"utf-8\").splitlines()))\n    check(\"Original audit population\", len(original_rows) == 208)\n    check(\"Original audit cells unique\",\n          len({(int(d[\"row\"]), int(d[\"column\"])) for d in original_rows}) == 208)\n    augmented = []\n    for d in original_rows:\n        r, c = int(d[\"row\"]), int(d[\"column\"])\n        check_label = 0 <= r < old.shape[0] and 0 <= c < old.shape[1] and inside[r, c]\n        if not check_label or int(d[\"gmw\"]) != int(old[r, c]):\n            raise ValueError(\"Original audit reference mismatch.\")\n        if abs(float(d[\"longitude\"]) - lon[r, c]) > 1e-7 or abs(float(d[\"latitude\"]) - lat[r, c]) > 1e-7:\n            raise ValueError(\"Original audit coordinate mismatch.\")\n        augmented.append(dict(d, gmw_v4112_2018=int(new[r, c]), worldcover_v100_2020=int(wc[r, c]),\n                              worldcover_class=WC_LABELS[int(wc[r, c])]))\n    check(\"Audit labels and coordinates match frozen grid\", True)\n    dump(out / \"audit_cells_with_context.json\", augmented)\n\n    case_register = json.loads((embedded / \"case_reviews_original.json\").read_text(encoding=\"utf-8\"))\n    check(\"Exactly three expected review cases\", {d[\"case\"] for d in case_register} == {\"C001\", \"C017\", \"C027\"} and len(case_register) == 3)\n    case_reports = []\n    for case in case_register:\n        rows = [d for d in augmented if d[\"cluster_id\"] == case[\"case\"]]\n        check(case[\"case\"] + \" original population\", len(rows) == {\"C001\": 11, \"C017\": 32, \"C027\": 42}[case[\"case\"]] == case[\"cells\"])\n        a = np.array([int(d[\"gmw\"]) for d in rows], dtype=\"uint8\")\n        b = np.array([d[\"gmw_v4112_2018\"] for d in rows], dtype=\"uint8\")\n        w = np.array([d[\"worldcover_v100_2020\"] for d in rows], dtype=\"uint8\")\n        case_reports.append({\"case\": case[\"case\"], \"original_review\": case,\n                             \"gmw_2018_reference_comparison\": compare_binary(a, b, np.ones(a.shape, bool)),\n                             \"worldcover_2020_later_context\": class_counts(w),\n                             \"cell_truth\": \"unresolved\", \"next_evidence\": case[\"next_evidence\"]})\n    summary = {\n        \"experiment\": EXPERIMENT, \"result_type\": \"Descriptive reference sensitivity; no new model scores\",\n        \"full_aoi_gmw2018\": compare_binary(old, new, inside),\n        \"full_aoi_worldcover2020\": class_counts(wc[inside]), \"cases\": case_reports,\n        \"limitations\": LIMITATIONS,\n    }\n    dump(out / \"comparison_summary.json\", summary)\n    dump(out / \"case_comparison.json\", case_reports)\n    diff = np.full(old.shape, 255, dtype=\"uint8\")\n    diff[inside] = old[inside] * 2 + new[inside]\n    write_grid(out / \"gmw_reference_pairs.tif\", diff, profile,\n               \"0=neither mapped;1=v4 only;2=v3 only;3=both;255=outside. Map differences, not ecological change.\")\n    render_maps(out, old, new, wc, inside, xs, ys, bounds, augmented, case_reports, native_cmap)\n    totals = summary[\"full_aoi_gmw2018\"]\n    lines = [\n        \"# Returned reference-context comparison\", \"\",\n        \"RESULT: Descriptive map sensitivity, not new model/field accuracy.\", \"\",\n        f\"Full AOI: {totals['centres']} frozen 20 m centres.\",\n        f\"GMW 2018: {totals['both_mapped']} mapped in both; {totals['v3_only_mapped']} v3-only; \"\n        f\"{totals['v4_only_mapped']} v4-only; {totals['both_not_mapped']} mapped in neither.\",\n        \"Differences are reference release/representation/sampling sensitivity, not confirmed 2018 ecological change.\", \"\",\n        \"| Original case | Centres | v3-only | v4-only | Both mapped |\",\n        \"| --- | ---: | ---: | ---: | ---: |\",\n    ]\n    for d in case_reports:\n        v = d[\"gmw_2018_reference_comparison\"]\n        lines.append(f\"| {d['case']} | {v['centres']} | {v['v3_only_mapped']} | {v['v4_only_mapped']} | {v['both_mapped']} |\")\n    lines += [\"\", \"WorldCover class histograms are in comparison_summary.json and case_comparison.json.\",\n              \"WorldCover zero/missing values are counted separately; no quality layer was supplied.\",\n              \"Original threshold, geographic split, predictions, audit categories and scores are unchanged.\", \"\",\n              \"## Limits\", \"\"] + [\"- \" + text for text in LIMITATIONS]\n    (out / \"RESULTS.md\").write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf-8\")\n    return summary\n\n\ndef render_maps(out, old, new, wc, inside, xs, ys, bounds, rows, cases, native_cmap):\n    import numpy as np\n    import os\n    # A Colab inline backend may be unavailable in the isolated interpreter.\n    # Set the environment before import: matplotlib.use() after import is too late.\n    os.environ[\"MPLBACKEND\"] = \"Agg\"\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap, BoundaryNorm\n    from matplotlib.patches import Patch\n    extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)\n    codes = sorted(WC_LABELS)\n    colors = [tuple(v / 255 for v in native_cmap.get(code, (110, 110, 110, 255))[:3]) for code in codes]\n    colors[0] = (0.9, 0.6, 0.9)\n    wc_index = np.zeros(wc.shape, dtype=\"uint8\")\n    for i, code in enumerate(codes):\n        wc_index[wc == code] = i\n    palettes = [ListedColormap([\"#e5e5e5\", \"#16835b\"])] * 2 + [ListedColormap(colors)]\n    norms = [BoundaryNorm([-.5, .5, 1.5], 2)] * 2 + [BoundaryNorm(np.arange(len(codes) + 1) - .5, len(codes))]\n    arrays = [old, new, wc_index]\n    titles = [\"GMW v3 — 2018\", \"GMW v4.1.12 — 2018\", \"WorldCover v100 — 2020 context\"]\n    present = sorted(set(int(v) for v in wc[inside]))\n    wc_legend = [Patch(facecolor=colors[codes.index(code)], label=str(code) + \": \" + WC_LABELS[code]) for code in present]\n    binary_legend = [Patch(facecolor=\"#e5e5e5\", label=\"Not mapped\"), Patch(facecolor=\"#16835b\", label=\"Mapped\")]\n    case_colors = {\"C001\": \"#aa00ff\", \"C017\": \"#e87700\", \"C027\": \"#080808\"}\n\n    def draw(name, selected=None):\n        fig, axes = plt.subplots(1, 3, figsize=(15, 7), layout=\"constrained\")\n        for i, ax in enumerate(axes):\n            ax.imshow(np.ma.array(arrays[i], mask=~inside), extent=extent,\n                      cmap=palettes[i], norm=norms[i], interpolation=\"nearest\")\n            for case, color in case_colors.items():\n                points = [d for d in rows if d[\"cluster_id\"] == case and (selected is None or case == selected)]\n                r = np.array([int(d[\"row\"]) for d in points], int)\n                c = np.array([int(d[\"column\"]) for d in points], int)\n                if len(r):\n                    ax.scatter(xs[r, c], ys[r, c], s=18, marker=\"s\", facecolors=\"none\", edgecolors=color, linewidths=.8, label=case)\n            if selected:\n                points = [d for d in rows if d[\"cluster_id\"] == selected]\n                r = np.array([int(d[\"row\"]) for d in points])\n                c = np.array([int(d[\"column\"]) for d in points])\n                ax.set_xlim(xs[r, c].min() - 100, xs[r, c].max() + 100)\n                ax.set_ylim(ys[r, c].min() - 100, ys[r, c].max() + 100)\n            handles, labels = ax.get_legend_handles_labels()\n            ax.legend(handles=handles + (wc_legend if i == 2 else binary_legend), fontsize=7, loc=\"upper left\")\n            ax.set_title(titles[i], fontsize=10)\n            ax.set_aspect(\"equal\")\n            ax.tick_params(labelsize=7)\n            ax.ticklabel_format(useOffset=False, style=\"plain\")\n            ax.set_xlabel(\"Easting (m), EPSG:32640\", fontsize=8)\n        axes[0].set_ylabel(\"Northing (m), EPSG:32640\", fontsize=8)\n        fig.suptitle((selected + \": \" if selected else \"\") + \"Reference context — not field truth or ecological change\", fontsize=12)\n        fig.savefig(out / name, dpi=150)\n        plt.close(fig)\n    draw(\"reference_context_overview.png\")\n    for case in cases:\n        draw(case[\"case\"] + \"_reference_context.png\", case[\"case\"])\n"
TEST_SOURCE = "\"\"\"Synthetic verification tests: executed only in the team's Colab runtime.\"\"\"\nimport tempfile\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nimport rasterio\nimport pyogrio\nimport geopandas as gpd\nfrom affine import Affine\nfrom shapely.geometry import box, Polygon\nfrom pyproj import Transformer\nfrom reference_context import compare_binary, class_counts, polygon_labels, sample_native, write_grid, verify_input, sha256\n\n\nclass ContextTests(unittest.TestCase):\n    def test_cross_tab(self):\n        d = compare_binary([0, 0, 1, 1], [0, 1, 0, 1], [True] * 4)\n        self.assertEqual([d[k] for k in (\"both_not_mapped\", \"v4_only_mapped\", \"v3_only_mapped\", \"both_mapped\")], [1] * 4)\n        self.assertAlmostEqual(d[\"mapped_jaccard\"], 1 / 3)\n\n    def test_masked_values_not_compared(self):\n        d = compare_binary([1, 255], [1, 99], [True, False])\n        self.assertEqual(d[\"centres\"], 1)\n        self.assertEqual(d[\"mapped_jaccard\"], 1)\n\n    def test_no_union_is_undefined(self):\n        self.assertIsNone(compare_binary([0], [0], [True])[\"mapped_jaccard\"])\n\n    def test_empty_population(self):\n        d = compare_binary([255], [255], [False])\n        self.assertEqual(d[\"centres\"], 0)\n        self.assertIsNone(d[\"differently_mapped_fraction\"])\n\n    def test_wrong_shape_rejected(self):\n        with self.assertRaises(ValueError):\n            compare_binary([[0]], [0], [True])\n\n    def test_nonbinary_rejected(self):\n        with self.assertRaises(ValueError):\n            compare_binary([0], [95], [True])\n\n    def test_worldcover_missing_separate(self):\n        rows = class_counts(np.array([0, 95, 95, 80], dtype=\"uint8\"))\n        self.assertEqual({d[\"code\"]: d[\"centres\"] for d in rows}, {0: 1, 80: 1, 95: 2})\n\n    def test_unknown_worldcover_code_rejected(self):\n        with self.assertRaises(ValueError):\n            class_counts(np.array([1]))\n\n    def test_polygon_boundary_included(self):\n        v = polygon_labels([box(0, 0, 1, 1)], np.array([0, .5, 2]), np.array([.5, .5, .5]))\n        np.testing.assert_array_equal(v, [1, 1, 0])\n\n    def test_empty_geometry_not_absence(self):\n        with self.assertRaises(ValueError):\n            polygon_labels([], np.array([.5]), np.array([.5]))\n\n    def test_invalid_geometry_rejected(self):\n        bowtie = Polygon([(0, 0), (1, 1), (0, 1), (1, 0), (0, 0)])\n        with self.assertRaises(ValueError):\n            polygon_labels([bowtie], np.array([.5]), np.array([.5]))\n\n    def test_native_categories_not_interpolated(self):\n        a = np.array([[10, 95], [80, 0]], dtype=\"uint8\")\n        tx = Affine(1, 0, 0, 0, -1, 2)\n        v, within = sample_native(a, tx, np.array([.5, 1.5, .5, 1.5, 2.5]),\n                                  np.array([1.5, 1.5, .5, .5, .5]))\n        np.testing.assert_array_equal(v, [10, 95, 80, 0, 0])\n        np.testing.assert_array_equal(within, [True, True, True, True, False])\n\n    def test_coordinate_order(self):\n        transformer = Transformer.from_crs(4326, 32640, always_xy=True)\n        x, y = transformer.transform(56.362, 24.996)\n        self.assertTrue(400000 < x < 500000 and 2700000 < y < 2800000)\n        lon, lat = Transformer.from_crs(32640, 4326, always_xy=True).transform(x, y)\n        np.testing.assert_allclose([lon, lat], [56.362, 24.996], atol=1e-8, rtol=0)\n\n    def test_grid_round_trip(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            path = Path(tmp) / \"grid.tif\"\n            a = np.array([[0, 1], [255, 1]], dtype=\"uint8\")\n            tx = Affine(20, 0, 435000, 0, -20, 2765700)\n            prof = dict(driver=\"GTiff\", width=2, height=2, crs=\"EPSG:32640\", transform=tx)\n            write_grid(path, a, prof, \"synthetic\")\n            with rasterio.open(path) as ds:\n                np.testing.assert_array_equal(ds.read(1), a)\n                self.assertEqual(ds.transform, tx)\n                self.assertEqual(ds.nodata, 255)\n\n    def test_geopackage_bbox(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            path = Path(tmp) / \"test.gpkg\"\n            gdf = gpd.GeoDataFrame({\"label\": [1, 2]}, geometry=[box(0, 0, 1, 1), box(10, 10, 11, 11)], crs=4326)\n            pyogrio.write_dataframe(gdf, path, layer=\"extent\", driver=\"GPKG\")\n            self.assertEqual(pyogrio.list_layers(path).tolist()[0][0], \"extent\")\n            local = pyogrio.read_dataframe(path, layer=\"extent\", columns=[], bbox=(.2, .2, .8, .8))\n            self.assertEqual(len(local), 1)\n\n    def test_checksum_gate(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            path = Path(tmp) / \"toy\"\n            path.write_bytes(b\"toy\")\n            verify_input(path, {\"size\": 3, \"sha256\": sha256(path)})\n            with self.assertRaises(ValueError):\n                verify_input(path, {\"size\": 3, \"sha256\": \"0\" * 64})\n\n\nif __name__ == \"__main__\":\n    unittest.main(verbosity=2)\n"
REQUIREMENTS = "numpy==2.1.3\nrasterio==1.4.4\ngeopandas==1.1.1\npyogrio==0.11.1\nshapely==2.1.2\npyproj==3.7.2\nmatplotlib==3.10.0\n"

try:
    for name, entry in EMBEDDED.items():
        payload = base64.b64decode(entry["base64"], validate=True)
        if hashlib.sha256(payload).hexdigest() != entry["sha256"]:
            raise ValueError("Embedded input hash mismatch: " + name)
        (ARTIFACTS / "embedded" / name).write_bytes(payload)
    (ARTIFACTS / "embedded_manifest.json").write_text(json.dumps(
        {name: {key: value for key, value in data.items() if key != "base64"}
         for name, data in EMBEDDED.items()}, indent=2) + "\n", encoding="utf-8")
    (ARTIFACTS / "reference_context.py").write_text(MODULE_SOURCE, encoding="utf-8")
    (ARTIFACTS / "test_reference_context.py").write_text(TEST_SOURCE, encoding="utf-8")
    (ARTIFACTS / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
    # Import is standard-library-only; geospatial imports occur later inside functions.
    sys.path.insert(0, str(ARTIFACTS))
    import importlib.util
    spec = importlib.util.spec_from_file_location("mp_reference_plan", ARTIFACTS / "reference_context.py")
    plan = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(plan)
    (ARTIFACTS / "METHOD.md").write_text(plan.METHOD, encoding="utf-8")
    (ARTIFACTS / "ATTRIBUTION.md").write_text(plan.ATTRIBUTION, encoding="utf-8")
    EXPECTED_UPLOADS = plan.EXPECTED
    STATUS["embedded_hashes_passed"] = True
    STATUS["status"] = "embedded_inputs_ready"
    print("Frozen embedded inputs restored; original audit population and grid will be checked in Colab.")
except Exception:
    STATUS["status"] = "failed_embedded_inputs"
    (ARTIFACTS / "error_embedded.txt").write_text(traceback.format_exc(), encoding="utf-8")
    print("Embedded-input checks failed; continue to the final cell for a diagnostic ZIP.")
save_status()


## 3. Install isolated, pinned dependencies

Revision bootstrap-fix-1 bypasses the failed `ensurepip` step: the environment is created without pip, then Colab's existing pip installs into that environment using `--python`. No global package replacement is performed. This approach is documented in [pip's interpreter guide](https://pip.pypa.io/en/stable/topics/python-option/).

If installation fails, the exact stage and logs are included in the diagnostic ZIP.


In [ ]:
if STATUS.get("embedded_hashes_passed"):
    try:
        import venv
        STATUS["installation_stage"] = "checking_host_pip"
        STATUS["host_python_version"] = sys.version
        pip_help = subprocess.run([sys.executable, "-m", "pip", "--help"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        (ARTIFACTS / "installer_preflight.log").write_text(pip_help.stdout, encoding="utf-8")
        if pip_help.returncode != 0 or "--python" not in pip_help.stdout:
            raise RuntimeError("The runtime needs pip 22.3+ with --python support. No global packages were changed.")
        VENV = RUN / "venv"
        STATUS["installation_stage"] = "creating_venv_without_pip"
        # Avoid ensurepip: it failed in the team's Python 3.13 runtime.
        venv.EnvBuilder(with_pip=False).create(VENV)
        PYTHON = VENV / "bin" / "python"
        PIP_FOR_VENV = [sys.executable, "-m", "pip", "--python", str(PYTHON)]
        STATUS["installation_stage"] = "installing_pinned_dependencies"
        save_status()
        cmd = PIP_FOR_VENV + ["install", "--only-binary=:all:",
                             "-r", str(ARTIFACTS / "requirements.txt")]
        print("Installing pinned libraries into this run's isolated environment...")
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        (ARTIFACTS / "installation.log").write_text(result.stdout, encoding="utf-8")
        STATUS["installation_returncode"] = result.returncode
        if result.returncode != 0:
            raise RuntimeError("Pinned installation failed; see installation.log.")
        STATUS["installation_stage"] = "verifying_target_environment"
        probe = subprocess.run([str(PYTHON), "-c",
            "import sys,json; print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix,'version':sys.version}))"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        (ARTIFACTS / "target_environment.log").write_text(probe.stdout, encoding="utf-8")
        if probe.returncode != 0:
            raise RuntimeError("Target environment probe failed.")
        target = json.loads(probe.stdout)
        if Path(target["prefix"]).resolve() != VENV.resolve() or target["prefix"] == target["base_prefix"]:
            raise RuntimeError("Target interpreter is not isolated; stop before analysis.")
        STATUS["target_python_version"] = target["version"]
        freeze = subprocess.run(PIP_FOR_VENV + ["freeze"],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        (ARTIFACTS / "installed_requirements.txt").write_text(freeze.stdout, encoding="utf-8")
        dep_check = subprocess.run(PIP_FOR_VENV + ["check"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        (ARTIFACTS / "dependency_check.log").write_text(dep_check.stdout, encoding="utf-8")
        STATUS["dependency_check_returncode"] = dep_check.returncode
        if freeze.returncode != 0 or dep_check.returncode != 0:
            raise RuntimeError("Dependency verification failed; see recorded installer logs.")
        DEPS_OK = True
        STATUS["status"] = "dependencies_ready"
        STATUS["installation_stage"] = "completed"
        print("Dependencies ready; the Colab kernel's installed packages were not changed.")
    except Exception as exc:
        DEPS_OK = False
        STATUS["status"] = "failed_installation"
        detail = traceback.format_exc()
        captured = getattr(exc, "output", None)
        if captured:
            detail += "\nCaptured subprocess output:\n" + (
                captured.decode("utf-8", errors="replace") if isinstance(captured, bytes) else str(captured))
        (ARTIFACTS / "error_installation.txt").write_text(detail, encoding="utf-8")
        print("Installation failed at:", STATUS.get("installation_stage", "unknown"))
        print(detail[-4000:])
        print("Continue to the final cell for a diagnostic ZIP.")
else:
    print("Installation skipped: embedded inputs failed.")
save_status()


## 4. Upload the two downloaded datasets

Select both files in one picker. Files must retain the exact names listed above. Original inputs are validated by size and SHA-256; no global source dataset is included in the results ZIP.

In [ ]:
if DEPS_OK:
    try:
        from google.colab import files
        original_cwd = Path.cwd()
        try:
            os.chdir(INPUTS)
            uploaded = files.upload()
        finally:
            os.chdir(original_cwd)
        upload_names = set(uploaded)
        uploaded.clear()
        del uploaded
        if upload_names != set(EXPECTED_UPLOADS):
            raise ValueError("Select exactly the two required files together, with their original filenames.")
        upload_manifest = {}
        for name, expected in EXPECTED_UPLOADS.items():
            path = INPUTS / name
            plan.verify_input(path, expected)
            upload_manifest[name] = dict(expected, verified=True)
        (ARTIFACTS / "uploaded_source_manifest.json").write_text(json.dumps(upload_manifest, indent=2) + "\n", encoding="utf-8")
        UPLOAD_OK = True
        STATUS["status"] = "uploaded_sources_verified"
        print("Both uploaded datasets passed size and checksum checks.")
    except Exception:
        STATUS["status"] = "failed_upload"
        (ARTIFACTS / "error_upload.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print("Upload checks failed. Continue to the final cell for a diagnostic ZIP, or rerun from cell 1.")
else:
    print("Upload skipped: dependencies or embedded inputs failed.")
save_status()


## 5. Verify, compare and download results

Synthetic tests run first. Analysis runs only if they pass. The ZIP contains methods, source/tests, hashes, maps and summaries—or failure logs. A success status is not independent ecological validation.

In [ ]:
def run_capture(command, log_name):
    result = subprocess.run(command, cwd=ARTIFACTS, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env={**os.environ, "MPLBACKEND": "Agg"})
    (ARTIFACTS / log_name).write_text(result.stdout, encoding="utf-8")
    print(result.stdout[-6000:])
    return result.returncode

if DEPS_OK and UPLOAD_OK:
    try:
        print("Running synthetic checks in the isolated Colab environment...")
        test_code = run_capture([str(PYTHON), "-m", "unittest", "-v", "test_reference_context"], "synthetic_tests.log")
        STATUS["synthetic_tests_returncode"] = test_code
        if test_code != 0:
            STATUS["status"] = "failed_synthetic_tests"
        else:
            print("Comparing maps on the frozen grid; no model fitting...")
            runner = (
                "from pathlib import Path\n"
                "from reference_context import run\n"
                f"run(Path({str(INPUTS)!r}), Path({str(ARTIFACTS / 'embedded')!r}), "
                f"Path({str(ARTIFACTS)!r}), Path({str(SCRATCH)!r}))\n"
            )
            (ARTIFACTS / "run_experiment.py").write_text(runner, encoding="utf-8")
            analysis_code = run_capture([str(PYTHON), str(ARTIFACTS / "run_experiment.py")], "analysis.log")
            STATUS["analysis_returncode"] = analysis_code
            STATUS["status"] = "completed_reference_context_comparison" if analysis_code == 0 else "failed_analysis"
    except Exception:
        STATUS["status"] = "failed_execution"
        (ARTIFACTS / "error_execution.txt").write_text(traceback.format_exc(), encoding="utf-8")
else:
    print("Analysis skipped; packaging diagnostic evidence.")

STATUS["packaging_revision"] = "rerun-safe-manifest-1"
STATUS["finished_utc"] = datetime.now(timezone.utc).isoformat()
save_status()
if STATUS["status"] == "completed_reference_context_comparison":
    summary = json.loads((ARTIFACTS / "comparison_summary.json").read_text(encoding="utf-8"))
    print(json.dumps(summary["full_aoi_gmw2018"], indent=2))
    from IPython.display import display, Image, Markdown
    display(Image(filename=str(ARTIFACTS / "reference_context_overview.png")))
    print("Three case zooms and their full summaries are included in the ZIP.")
else:
    print("Run did not complete successfully. Return the diagnostic ZIP; do not treat partial files as results.")
    (ARTIFACTS / "READ_FAILURE_FIRST.md").write_text(
        "# Incomplete run\n\nRead run_status.json and logs. Partial maps or summaries are NOT accepted results.\n",
        encoding="utf-8")

def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Own-run files only; never package uploads, scratch global GeoPackage, environment or other accounts.
manifest_exclusions = {"SHA256SUMS.json"}
if STATUS["status"] == "completed_reference_context_comparison":
    manifest_exclusions.add("READ_FAILURE_FIRST.md")
artifact_files = sorted(path for path in ARTIFACTS.rglob("*")
                        if path.is_file() and "__pycache__" not in path.parts
                        and path.name not in manifest_exclusions)
manifest = {str(path.relative_to(ARTIFACTS)).replace("\\", "/"): {
                "size": path.stat().st_size, "sha256": file_sha(path)}
            for path in artifact_files}
(ARTIFACTS / "SHA256SUMS.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
ZIP = RUN / (RUN.name + "_results.zip")
with zipfile.ZipFile(ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_files + [ARTIFACTS / "SHA256SUMS.json"]:
        archive.write(path, arcname=str(path.relative_to(ARTIFACTS)))
with zipfile.ZipFile(ZIP) as archive:
    bad = archive.testzip()
    if bad:
        raise ValueError("ZIP CRC failure: " + bad)
print("Status:", STATUS["status"])
print("ZIP:", ZIP.name, "bytes:", ZIP.stat().st_size, "SHA-256:", file_sha(ZIP))
from IPython.display import FileLink, display
display(FileLink(str(ZIP)))
try:
    from google.colab import files
    files.download(str(ZIP))
except Exception:
    print("Automatic download unavailable. Use the link above or the Colab Files panel.")


## Reading the returned ZIP

Start with `run_status.json`. Only `completed_reference_context_comparison` is a completed computational run.

Read `RESULTS.md`, `comparison_summary.json` and `case_comparison.json`; inspect the overview and the three case zoom PNGs. Zero/missing WorldCover values are explicitly listed. The augmented audit is JSON and preserves the original fields alongside new map context.

`METHOD.md`, `ATTRIBUTION.md`, `environment.json`, `data_checks.json`, `synthetic_tests.log` and `SHA256SUMS.json` explain the method and verification. The manifest intentionally does not hash itself. Upload/source hashes identify the large excluded originals; small frozen inputs are included.

The comparison does not replace independent, dated cover evidence or field inspection.
